An off-lattice particle-cluster polydisperse tunable sequential aggregation (PTSA) algorithm by Singh and Tsotas 2022, https://doi.org/10.1016/j.ces.2021.117022  

In [13]:
import numpy as np
import numba
import h5py
import pandas as pd
import os
import time
import glob
from datetime import date

In [14]:
@numba.njit
def ptsa(Np, mean_rp, rel_std_rp, Df, k, max_search_num, rng):
    """
    Polydisperse tunable sequential aggregation (PTSA) method for generating fractal-like aggregate.
    Monomer radius follows a normal distribution with specified mean and standard deviation.
    The algorithms might fail depending on the combination of (k, Df).
    N.Moteki tested only k=0.95, and Df= 2.35~2.95, 

    Inner distance-check loop uses early exit on overlap detection,
    avoiding full O(ip) distance scans for rejected candidates.

    ==== Input parameters ===
    Np: number of monomers
    mean_rp: mean of the normal distribution of the monomer radius
    rel_std_rp: relative standard deviation of the normal distrituion of the monomer radius 
    Df: fractal dimension
    k: fractal prefactor
    max_search_num: maximum number of iteration for searching a location of each monomer attached onto the surface of an aggregate (= 20000000)
    rng: random number generator constructed by the numpy.random.default_rng()

    === Output variables ===
    Np_final: number of monomers in the generated aggregate. If the aggregate generation failed, Np_final < Np.
    xp: (x,y,z) coordinate of the center of individual monomers relative to the centroid of the aggregate. 2D of shape= (Np,3).
    rp: radii of individual monomers. 1D array of size= Np.
    V: volume of the aggregate.
    Rve: volume-equivalent radius of the aggregate.
    Rg: gyration radius of aggregate.
    eps_agg: porosity of the aggregate calculated using the gyration method.

    reference: 
    Singh and Tsotas 2022, https://doi.org/10.1016/j.ces.2021.117022  

    Coded and tested by Nobuhiro Moteki, 2023-Jan-29th
    Optimized by Claude (early-exit distance loop), 2026-Mar-09th
    """

    #torelance of inter-monomer separation between attached monomers
    tol = 0.001+(2.95-Df)*0.0033 #tol= 0.001 @ Df=2.95, tol=0.003 @ Df=2.35

    # box muller method for genarating standard normal random vairable snv
    u1= rng.random(size= Np)
    u2= rng.random(size= Np)
    r= np.sqrt(-2.0*np.log(u1))
    theta= 2.0*np.pi*u2
    snv= r*np.cos(theta) # random number from the standard normal distribution

    # enforce outliers beyond 2 sigma to the boundary values
    for i, val in enumerate(snv):
        if val < -2:
            snv[i]= -2
        elif val > 2:
            snv[i]= 2

    rp= mean_rp*rel_std_rp*snv + mean_rp # radius of indivitual monomers (normal distribution) 
    density = 1.0 # monomer bulk density
    mp= (4*np.pi/3)*density*rp**3  # mass of individual monomers
    mp3t= np.broadcast_to(mp, (3,Np))
    mp3= mp3t.transpose() 
    xp= np.zeros((Np,3)) # 3D coordinates (x,y,z) of the centers of Np monomers
    minimum_separation= mean_rp*1e-6  # tolerance of intermonomer surface separation [um]

    xp[0,0],xp[0,1],xp[0,2]= 0.0,0.0,0.0
    # generate random point on a unit surface
    phi= rng.random()*2*np.pi
    u= rng.random()*2-1.0
    sqrt1mu2= np.sqrt(1-u**2)
    ux,uy,uz= sqrt1mu2*np.cos(phi), sqrt1mu2*np.sin(phi), u   # random position on the surface of the unit sphere
    R= rp[0]+rp[1]+minimum_separation # distance between the centers of 1st and 2nd monomers
    xp[1,0],xp[1,1],xp[1,2]= R*ux, R*uy, R*uz  # center position of 2nd monomer
    xp[0:2,:]= xp[0:2,:]- np.sum(mp3[0:2,:]*xp[0:2,:],axis=0)/np.sum(mp[0:2]) # shift the centroid to the origin 

    Np_final=0
    for ip in range(2,Np):
        n= ip+1 # index of monomer (3,...,Np)
        Rpn= rp[ip] # radius of n-th monomer
        if n <= k**(3/(3-Df)):
            Df_tuned= 3
        else:
            Df_tuned= Df*(np.log(Np)/np.log(Np/k))  # Step 4. Compute tuned fractal dimension (at k=1) from the actual fractal dimensiton and prefactor (Df, k)
        R2= (n**2*Rpn**2/(n-1))*n**(2/Df_tuned)- (n*Rpn**2/(n-1))- n*Rpn**2*(n-1)**(2/Df_tuned)
        R= np.sqrt(R2)

        #maximum_separation= mean_rp*tol
        maximum_separation= mean_rp*tol*(1 + 5*(ip/Np)*(2.95-Df)/0.6)  # Np and Df dependent tuning for accerelation

        attached_cond= False
        for i_search in range(max_search_num):

            phi= rng.random()*2*np.pi
            u= rng.random()*2-1.0
            sqrt1mu2= np.sqrt(1-u*u)
            x0= R*sqrt1mu2*np.cos(phi)
            x1= R*sqrt1mu2*np.sin(phi)
            x2= R*u  # candidate center location of n-th monomer

            # Early-exit overlap and proximity check.
            # Iterates monomers one by one; breaks immediately on overlap (avoids full O(ip) scan).
            overlapping= False
            proximate= False
            for j in range(ip):
                dx= xp[j,0]-x0
                dy= xp[j,1]-x1
                dz= xp[j,2]-x2
                surf_dist= np.sqrt(dx*dx+dy*dy+dz*dz)-(rp[j]+Rpn)
                if surf_dist < minimum_separation:
                    overlapping= True
                    break  # early exit: overlap found, no need to check remaining monomers
                if surf_dist < maximum_separation:
                    proximate= True

            if overlapping:
                continue

            if proximate:
                attached_cond= True
                xp[ip,0]= x0
                xp[ip,1]= x1
                xp[ip,2]= x2
                xp[0:n,:]= xp[0:n,:]- np.sum(mp3[0:n,:]*xp[0:n,:],axis=0)/np.sum(mp[0:n]) # shift the centroid to the origin
                break

        Np_final=n
        if not attached_cond:
            break 
         
    if Np_final == Np:
        Rg= np.sqrt((1/np.sum(mp[:]))*np.sum(mp3[:,0]*((xp[:,0]**2+ xp[:,1]**2+ xp[:,2]**2) +3/5*rp[:]**2))) # Gyration radius
        Re= np.sqrt(5/3)*Rg # effectuve radius
        Vagg= 4*np.pi/3*Re**3 # effective volume
        V= np.sum(mp[:])/density # true volume
        Rve= np.cbrt(3*V/(4*np.pi)) # volume equivalent radius
        eps_agg= 1- V/Vagg # porosity
    else:
        Rg= 0
        Re= 0
        Vagg= 0
        V= 0
        Rve= 0
        eps_agg= 0
    
    return Np_final, xp, rp, V, Rve, Rg, eps_agg

In [15]:
### pre-execution of ptsa for njit compilation (User should not edit this cell) ### 

mean_rp = 0.015 # mean monomer radius [um]  (user input)
rel_std_rp = 0.1 # relative standard deviation of monomer radius [-]  (user input)
Df =2.9 # fractal dimension (Df< 2.95)
k =0.9 # fractal prefactor (k<= 0.95)
max_search_num= 1000000

Np=20
rng = np.random.default_rng()
Np_final, xp, rp, V, Rve, Rg, eps_agg = ptsa(Np, mean_rp, rel_std_rp, Df, k, max_search_num, rng)

In [ ]:
mean_rp = 0.015 # mean monomer radius [um]  (user input)
rel_std_rp = 0.1 # relative standard deviation of monomer radius [-]  (user input)

# range of the number of monomers
Np_min= 350
Np_max= 2800
num_Np= 2  # Number of Np grid points between Np_min and Np_max (linspace)

# range of the fractal dimension
Df_min= 2.35 #(default)
Df_max= 2.95 #(default)
num_Df= 13  # Number of Df grid points between Df_min and Df_max (linspace)

k =0.95 # fractal prefactor 

# index list of random aggregates
agg_num_arr= [0,1,2,3,4,5,6,7,8,9]

In [17]:
Np_arr= np.linspace(start=Np_min, stop=Np_max, num=num_Np).astype(int)
Np_arr

array([ 350, 2800])

In [ ]:
### execution of the njitted ptsa — output written to HDF5 ###

max_search_num = 1000000000
max_try_num    = 20

out_dir  = "./generated_agg_files/"

# ---- determine filenames: aggregates_YYYYMMDD_xx.h5 / catalog_YYYYMMDD_xx.csv ----
date_str      = date.today().strftime("%Y%m%d")
existing      = sorted(glob.glob(os.path.join(out_dir, f"aggregates_{date_str}_??.h5")))
next_id       = len(existing)       # 00 for the first run today, 01 for the second, ...
h5_fname      = f"aggregates_{date_str}_{next_id:02d}.h5"
catalog_fname = f"catalog_{date_str}_{next_id:02d}.csv"
h5_path       = os.path.join(out_dir, h5_fname)
catalog_path  = os.path.join(out_dir, catalog_fname)
print(f"HDF5 output  : {h5_path}")
print(f"Catalog      : {catalog_path}")

Df_arr = np.linspace(start=Df_min, stop=Df_max, num=num_Df)
Np_arr = np.linspace(start=Np_min, stop=Np_max, num=num_Np).astype(int)

def make_h5key(agg_num, k, Df, mean_rp, rel_std_rp, Np):
    """HDF5 group path that uniquely identifies an aggregate by its input parameters.
    Format: '{agg_num}/{k:.3f}/{Df:.2f}/{mean_rp:.4f}/{rel_std_rp:.2f}/{Np:05d}'
    """
    return f"{agg_num}/{k:.3f}/{Df:.2f}/{mean_rp:.4f}/{rel_std_rp:.2f}/{Np:05d}"

new_records = []

os.makedirs(out_dir, exist_ok=True)

with h5py.File(h5_path, 'a') as h5f:
    for agg_num in agg_num_arr:
        for Df in Df_arr:
            for Np in Np_arr:
                key = make_h5key(agg_num, k, Df, mean_rp, rel_std_rp, Np)

                if key in h5f:
                    print(f"Skip (already exists): {key}", flush=True)
                    continue

                for i_try in range(max_try_num):
                    print(f"agg_num={agg_num:02d}, Df={Df:.2f}, Np={Np:05d}, i_try={i_try:02d}", flush=True)
                    rng = np.random.default_rng()
                    t0  = time.perf_counter()
                    Np_final, xp, rp, V, Rve, Rg, eps_agg = ptsa(
                        Np, mean_rp, rel_std_rp, Df, k, max_search_num, rng)
                    elapsed_min = (time.perf_counter() - t0) / 60
                    print(f"  Np_final={Np_final:05d}, {elapsed_min:.2f} min", flush=True)

                    if Np_final != Np:
                        if i_try < max_try_num - 1:
                            continue          # retry
                        print(f"  FAILED after {max_try_num} tries: {key}")
                        break

                    # ---- success: write to HDF5 ----
                    grp = h5f.create_group(key)
                    grp.create_dataset("xp", data=xp,
                                       compression="gzip", compression_opts=4)
                    grp.create_dataset("rp", data=rp,
                                       compression="gzip", compression_opts=4)
                    grp.attrs.update({
                        "agg_num":     int(agg_num),
                        "k":           float(k),
                        "Df":          float(round(Df, 2)),
                        "mean_rp":     float(mean_rp),
                        "rel_std_rp":  float(rel_std_rp),
                        "Np":          int(Np),
                        "Rve":         float(Rve),
                        "Rg":          float(Rg),
                        "eps_agg":     float(eps_agg),
                    })
                    new_records.append({
                        "agg_num":     int(agg_num),
                        "k":           float(k),
                        "Df":          float(round(Df, 2)),
                        "mean_rp":     float(mean_rp),
                        "rel_std_rp":  float(rel_std_rp),
                        "Np":          int(Np),
                        "Rve":         float(Rve),
                        "Rg":          float(Rg),
                        "eps_agg":     float(eps_agg),
                        "h5_file":     h5_fname,
                        "h5_key":      key,
                    })
                    print(f"  Saved: {key}", flush=True)
                    break

# ---- write catalog CSV (new file per run, matching the HDF5 filename) ----
if new_records:
    df_catalog = pd.DataFrame(new_records)
    df_catalog.to_csv(catalog_path, index=False)
    print(f"\nCatalog saved: {catalog_path} ({len(new_records)} entries).")
else:
    print("No new aggregates were generated.")

HDF5 output  : ./generated_agg_files/aggregates_20260314_01.h5
Catalog      : ./generated_agg_files/catalog_20260314_01.csv
agg_num=00, Df=2.35, Np=00350, i_try=00
  Np_final=00350, 0.00 min
  Saved: 0/0.950/2.35/0.0150/0.10/00350
agg_num=00, Df=2.35, Np=02800, i_try=00
  Np_final=02800, 0.16 min
  Saved: 0/0.950/2.35/0.0150/0.10/02800
agg_num=00, Df=2.95, Np=00350, i_try=00
  Np_final=00003, 0.05 min
agg_num=00, Df=2.95, Np=00350, i_try=01
  Np_final=00350, 0.00 min
  Saved: 0/0.950/2.95/0.0150/0.10/00350
agg_num=00, Df=2.95, Np=02800, i_try=00
  Np_final=00003, 0.06 min
agg_num=00, Df=2.95, Np=02800, i_try=01
  Np_final=02800, 0.22 min
  Saved: 0/0.950/2.95/0.0150/0.10/02800
agg_num=01, Df=2.35, Np=00350, i_try=00
  Np_final=00350, 0.00 min
  Saved: 1/0.950/2.35/0.0150/0.10/00350
agg_num=01, Df=2.35, Np=02800, i_try=00
  Np_final=02800, 0.15 min
  Saved: 1/0.950/2.35/0.0150/0.10/02800
agg_num=01, Df=2.95, Np=00350, i_try=00
  Np_final=00003, 0.05 min
agg_num=01, Df=2.95, Np=00350, i_

## Export to MSTM Input Format (.ptsa)

To convert a generated aggregate to the MSTM input format (`.ptsa` CSV), use the separate notebook **`export_ptsa_from_hdf5.ipynb`**.

### Steps

1. Check `catalog_YYYYMMDD_xx.csv` to confirm the parameters (`agg_num`, `k`, `Df`, `mean_rp`, `rel_std_rp`, `Np`) of the aggregate to export.
2. Open `export_ptsa_from_hdf5.ipynb`, edit the parameters in cell-3, and run it.

### Output file naming convention

| File | Naming rule |
| --- | --- |
| HDF5 data | `aggregates_YYYYMMDD_xx.h5` |
| Catalog | `catalog_YYYYMMDD_xx.csv` |

`YYYYMMDD` is the execution date; `xx` is the sequential run index within the day (00, 01, 02, ...).  
The HDF5 file and catalog with the same `xx` always correspond to the same run.

### .ptsa file format

Each line contains the coordinates and radius of one monomer, space-delimited:

```
{x [um]}  {y [um]}  {z [um]}  {r_pp [um]}
```

The filename contains **input parameters only** (computed properties are stored in HDF5 group attributes and `catalog_YYYYMMDD_xx.csv`):

```
agg_num={n}_k={k:.3f}_Df={Df:.2f}_meanRp={mean_rp:.3f}um_rstdRp={rel_std_rp:.2f}_Np={Np:05d}.ptsa
```